Sensitivity of DEA results to the EC distance cutoff and the NeuN-low threshold. Here, the DEA tables (07) are compared across cutoffs for the genes of interest plus the top hits. Heatmaps: color = log2FC from group means (`_scanpy`), text = MAST FDR. Per set, the heatmap genes are exported to xlsx.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"font.sans-serif": ["Arial", "DejaVu Sans"], "mathtext.default": "regular", "svg.fonttype": "none"})
%config InlineBackend.figure_format = "retina"

In [ ]:
model = "MAST-RE"  # MAST-RE[-CDR]
cap = "5k"  # max cells per condition
mb = f"{model}-slide-joint_{cap}"  # between-sample (ECKO vs Ctrl): slide covariate, fitted on all genotypes
runs = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark/analysis/foxf2/Cellpose_1_nuclei_model/dea") / f"{model}_{cap}"
out = runs / "sensitivity"
out.mkdir(exist_ok=True)
pcts = [5, 10, 15, 20, 25]
sfx = {p: "" if p == 10 else f"_lowNeuN-p{p}" for p in pcts}
genes_neuron = ["Acsl4", "Gclc", "Gclm", "Gpx4", "Slc7a11", "Slc11a2", "Tfrc", "Trf"]
genes_ec = [
    "Foxf2", "Tek", "Vegfa", "Vegfb", "Cldn5", "Cdh5", "Tjp1", "Tjp2",
    "Nos3", "Sod1", "Cat", "Gclm", "Gclc", "Slc7a11", "Slc3a2", "Abcg2",
]

# label: (sheet, genes, reference cutoff, {cutoff: file})
sets = {
    "distance_proximal-vs-distal-ECs-in-ECKO": ("ECs", genes_ec, 18, {
        d: f"foxf2_ECKO-ECs-{d}um-from-lowNeuN_vs_ECKO-other-ECs_cortex_{model}_{cap}.xlsx" for d in [18, 31, 58, 103]
    }),
    "lowNeuN_lowNeuN-in-ECKO-vs-Ctrl-Glut": ("Neurons_Glut", genes_neuron, 10, {
        p: f"foxf2_ECKO-lowNeuN_vs_WT-Glut_cortex_{mb}{sfx[p]}.xlsx" for p in pcts
    } | {100: f"foxf2_ECKO_vs_WT_cortex_{mb}.xlsx"}),  # 100 = all ECKO Glut neurons
    "lowNeuN_lowNeuN-vs-normalNeuN-in-ECKO": ("Neurons_Glut", genes_neuron, 10, {
        p: f"foxf2_ECKO-lowNeuN_vs_ECKO-normalNeuN_cortex_{model}_{cap}{sfx[p]}.xlsx" for p in pcts
    }),
    "lowNeuN_proximal-vs-distal-ECs-in-ECKO-18um": ("ECs", genes_ec, 10, {
        p: f"foxf2_ECKO-ECs-18um-from-lowNeuN_vs_ECKO-other-ECs_cortex_{model}_{cap}{sfx[p]}.xlsx" for p in pcts
    }),
    "lowNeuN_proximal-ECs-in-ECKO-vs-Ctrl-ECs-18um": ("ECs", genes_ec, 10, {
        p: f"foxf2_ECKO-ECs-18um-from-lowNeuN_vs_WT-ECs_cortex_{mb}{sfx[p]}.xlsx" for p in pcts
    }),
}

In [ ]:
for label, (sheet, genes, ref, files) in sets.items():
    files = {k: f for k, f in files.items() if (runs / f).exists()}
    if ref not in files:
        print(f"=== {label}: skipped, available cutoffs: {list(files)}")
        continue
    df = pd.concat({k: pd.read_excel(runs / f, sheet_name=sheet) for k, f in files.items()}, names=["cutoff", None]).reset_index(level=0)
    df = df.rename(columns={c: f"{c}_MAST" for c in ["FC", "log2FC", "PValue", "FDR"]})
    df["log2FC_scanpy"] = np.log2((np.expm1(df["mean_test"]) + 1e-9) / (np.expm1(df["mean_ref"]) + 1e-9))
    df["FC_scanpy"] = 2 ** df["log2FC_scanpy"]
    lfc = df.pivot(index="gene", columns="cutoff", values="log2FC_scanpy")[list(files)]
    fdr = df.pivot(index="gene", columns="cutoff", values="FDR_MAST")[list(files)]
    sig = fdr < 0.05

    summ = df.groupby("cutoff")[["design", "n_cells_test", "n_cells_ref"]].first().assign(
        n_sig=sig.sum(),
        n_up=(sig & (lfc > 0)).sum(),
        n_down=(sig & (lfc < 0)).sum(),
        spearman_vs_ref=lfc.corrwith(lfc[ref], method="spearman"),
        same_sign_ref_sig=np.sign(lfc).eq(np.sign(lfc[ref]), axis=0)[sig[ref]].mean()
    )
    genes = [x for x in genes if x in lfc.index]
    n_goi = len(genes)
    genes += fdr[ref].drop(genes).nsmallest(5).index.tolist()
    df.set_index(["gene", "cutoff"]).loc[genes, ["FC_MAST", "log2FC_MAST", "PValue_MAST", "FDR_MAST", "FC_scanpy", "log2FC_scanpy"]].to_excel(
        out / f"sensitivity_{label}.xlsx", merge_cells=False)
    print(f"=== {label}")
    display(summ.round(3))

    m = lfc.loc[genes]
    vmax = np.nanmax(np.abs(m.values))
    fig, ax = plt.subplots(figsize=(0.6 * len(files) + 1.5, 0.2 * len(genes) + 0.8))
    im = ax.imshow(m.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    for i, gene in enumerate(genes):
        for j, k in enumerate(files):
            q = fdr.loc[gene, k]
            if pd.notna(q):
                ax.text(j, i, f"{q:.3f}" if q >= 0.001 else "<0.001", ha="center", va="center", fontsize=6,
                        fontweight="bold" if q < 0.05 else "normal", color="w" if abs(m.loc[gene, k]) > 0.6 * vmax else "k")
    ax.axhline(n_goi - 0.5, color="k", lw=0.5)
    ax.set_xticks(range(len(files)), list(files))
    ax.set_yticks(range(len(genes)), genes, fontsize=7)
    ax.set_xlabel("Proximal distance cutoff (µm)" if label.startswith("distance") else "NeuN$^{low}$ percentile (Ctrl mice)")
    ax.set_title(f"{label.split('_', 1)[1].replace('-', ' ')}\nvalues: FDR", fontsize=8)
    ax.spines[:].set_visible(False)
    ax.tick_params(length=0)
    fig.colorbar(im, ax=ax, shrink=0.5, label="log2FC").outline.set_visible(False)
    fig.savefig(out / f"sensitivity_{label}.png", bbox_inches="tight", dpi=200)
    plt.show()